# Anime Recommender Inference Test 
Here we can test creating user representations from arbitrary anime ID lists and prompts.

In [12]:
import pickle
import torch
import torch.nn.functional as F
import numpy as np
from attentionrec.models.attentionrec import TransformerRecommendationModel
import os
import warnings
from tqdm import TqdmWarning

warnings.filterwarnings("ignore", category=TqdmWarning)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_ROOT = os.getcwd()  # current working dir
model_path = os.path.join(PROJECT_ROOT, "checkpoints/attentionrec/latest_checkpoint.pt")
embeddings_path = os.path.join(PROJECT_ROOT, "data/processed/attentionrec/anime-embeddings.pkl")

# %%
# Load embeddings
with open(embeddings_path, "rb") as f:
    emb_data = pickle.load(f)
    
description_embeddings = torch.tensor(emb_data['embeddings'], device=device)
embedding_dim = emb_data['embedding_dim']
anime_id_to_idx = emb_data['anime_id_to_idx']
idx_to_anime_id = {v: k for k, v in anime_id_to_idx.items()}

# %%
# Load model
model = TransformerRecommendationModel(
    embedding_dim=embedding_dim,
    num_heads=4,
    num_layers=2,
    dropout_rate=0.1,
).to(device)

# Explicitly allow unsafe globals (full unpickling)
checkpoint = torch.load(model_path, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Model loaded successfully.")

# Function to create a "user" embedding from a list of anime IDs
def create_user_embedding(anime_id_list):
    """
    anime_id_list: list of anime IDs to build the user representation from
    B: number of items to sample if len(list) > B
    """
    # Filter out anime IDs not in embeddings
    valid_ids = [i for i in anime_id_list if i in anime_id_to_idx]
    if not valid_ids:
        raise ValueError("No valid anime IDs found in embeddings!")

    indices = [anime_id_to_idx[i] for i in valid_ids]
    user_embeddings = description_embeddings[indices]
    print(f"User embedding shape before model: {user_embeddings.shape}")
    user_rep = model(user_embeddings.unsqueeze(0)).squeeze()
    return user_rep

# Function to compute top-K recommendations given a user embedding
def recommend_topk(user_rep, K=10, mask_ids=None):
    """
    user_rep: tensor [embedding_dim]
    K: number of recommendations
    mask_ids: optional list of anime IDs to mask (e.g., already seen)
    """
    scores = user_rep @ description_embeddings.T  # [num_items]

    # Mask any IDs
    if mask_ids:
        mask_idx = [anime_id_to_idx[i] for i in mask_ids if i in anime_id_to_idx]
        scores[mask_idx] = -float("inf")

    topk_idx = torch.topk(scores, K).indices.cpu().numpy()
    topk_anime_ids = [idx_to_anime_id[i] for i in topk_idx]
    return topk_anime_ids

def create_augmented_user_embedding(anime_id_list, prompt_embedding=None, prompt_weight=2.0):
    valid_ids = [i for i in anime_id_list if i in anime_id_to_idx]
    
    if len(valid_ids) != len(anime_id_list):
        print(f"Warning: {len(anime_id_list) - len(valid_ids)} anime IDs were not found in embeddings and will be ignored.")

    indices = [anime_id_to_idx[i] for i in valid_ids]
    user_item_embeddings = description_embeddings[indices]  # [num_items, D9
    if user_item_embeddings.dim() == 2:
        user_item_embeddings = user_item_embeddings.unsqueeze(0)  # (1, num_items, D)

    if prompt_embedding is not None:
        if prompt_embedding.dim() == 1:
            prompt_embedding = prompt_embedding.unsqueeze(0)  # (1, D)
        elif prompt_embedding.dim() > 2:
            raise ValueError(f"Prompt embedding has too many dimensions: {prompt_embedding.shape}")

        prompt_embedding_scaled = prompt_embedding.unsqueeze(1) * prompt_weight

        combined_embeddings = torch.cat([user_item_embeddings, prompt_embedding_scaled], dim=1)  # (1, num_items+1, D)
    else:
        combined_embeddings = user_item_embeddings  # (1, num_items, D)
    assert combined_embeddings.dim() == 3, f"Combined embeddings must be 3D, got {combined_embeddings.shape}"

    with torch.no_grad():
        user_rep = model(combined_embeddings).squeeze(0)  # remove batch dim

    return user_rep

Model loaded successfully.


In [3]:
#load anime csv to get the title
import pandas as pd
csv_path = "data/raw/mal/anime-dataset-2023.csv"

# Read CSV (assuming columns include 'anime_id' and 'title')
df = pd.read_csv(csv_path)

# Create a mapping from ID to title
id2title = dict(zip(df['anime_id'], df['Name']))

# Synthetic user
We create a synthetic user with a few genre defining items, and then see how a prompt can augment this list.

In [29]:
archetype_users = {
    "shounen_watcher": [20, 21, 5114, 813],
    "slice_of_life_fan": [5680, 10165, 34798],
    "psych_fan": [1535, 9253],
    "game_strategist": [19815, 17265, 34933, 62795],
    "classic_veteran": [1, 97, 467],
}


for u, anime_ids_for_user in archetype_users.items():
    print(u)
    
    # Create user embedding
    user_rep = create_user_embedding(anime_ids_for_user)
    
    # Get top-K recommendations
    top10 = recommend_topk(user_rep, K=10, mask_ids=anime_ids_for_user)
    print("Top-10 recommendations for synthetic user:", top10)
    
    # Get titles for your list of IDs
    titles = [id2title.get(aid, f"Unknown ID {aid}") for aid in top10]
    print(titles)


shounen_watcher
User embedding shape before model: torch.Size([4, 1024])
Top-10 recommendations for synthetic user: [121, 1535, 269, 225, 791, 39417, 32379, 223, 1292, 55453]
['Fullmetal Alchemist', 'Death Note', 'Bleach', 'Dragon Ball GT', 'Arion', 'Granbelm', 'Berserk', 'Dragon Ball', 'Afro Samurai', 'Naruto (2023)']
slice_of_life_fan
User embedding shape before model: torch.Size([3, 1024])
Top-10 recommendations for synthetic user: [17082, 17549, 18495, 39026, 853, 7791, 29787, 14967, 1852, 3604]
['Aiura', 'Non Non Biyori', 'Kitaku-bu Katsudou Kiroku', 'Dumbbell Nan Kilo Moteru?', 'Ouran Koukou Host Club', 'K-On!!', 'Gochuumon wa Usagi desu ka??', 'Boku wa Tomodachi ga Sukunai Next', 'Hidamari Sketch', 'Hidamari Sketch x 365']
psych_fan
User embedding shape before model: torch.Size([2, 1024])
Top-10 recommendations for synthetic user: [31580, 28791, 6547, 36882, 38668, 49956, 384, 44200, 16890, 53012]
['Ajin', 'Gunslinger Stratos The Animation', 'Angel Beats!', 'Arifureta Shokugyou 

# Augmented user embedding with a prompt
Because we used SentenceTransformer embeddings to train,
we can use natural language prompts to generate user embeddings without needing to specify anime IDs. 
This is a powerful feature of using text-based embeddings and attention mechanisms, as it allows us to capture user preferences in a more flexible way.

We can see that the model starts recommending more "magical girl" anime after augmenting the user recommendation with a prompt.

In [24]:
archetype_users = {
    "prompt_only_user": [],  # This user has no anime IDs, so their embedding will be based solely on the prompt.
    "shounen_watcher": [20, 21, 5114, 813],
    "slice_of_life_fan": [5680, 10165, 34798],
    "psych_fan": [1535, 9253],
    "game_strategist": [19815, 17265, 34933, 62795],
    "classic_veteran": [1, 97, 467],
}

valid_ids = set(idx_to_anime_id.keys())

archetype_users = {
    user: [i for i in ids if i in valid_ids]
    for user, ids in archetype_users.items()
}


In [27]:
from sentence_transformers import SentenceTransformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "stsb-roberta-large" # We need to use the same model as the one used to generate the description embeddings, otherwise the embeddings won't be compatible with the model's attention layers.


# Load model
print(f"Loading model {model_name}...")
embedding_model = SentenceTransformer(model_name)
embedding_model = embedding_model.to(device)

# Because we used SentenceTransformer embeddings to train,
# we can in theory use natural language prompts to generate user embeddings without needing to specify anime IDs. 
# This is a powerful feature of using text-based embeddings and attention mechanisms, as it allows us to capture user preferences in a more flexible way.

# Prepare prompt
# You can experiment with different prompts to see how they influence the recommendations. The prompt should ideally capture the essence of the user's preferences or archetype. 
prompt = "A young girl gains magical powers and balances school life with fighting supernatural threats."  # Example prompt describing a user archetype. In practice, you could have more complex prompts or even multiple prompts per user.
prompt_embedding = embedding_model.encode(prompt, normalize_embeddings=True, show_progress_bar=True, convert_to_tensor=True)  # SentenceTransformer


print(f"prompt: {prompt}")
for u, anime_ids_for_user in archetype_users.items():
    print(u)

    # Create augmented user embedding
    user_rep = create_augmented_user_embedding(anime_ids_for_user, prompt_embedding=prompt_embedding, prompt_weight=4.0)

    # Get top-K recommendations
    top10 = recommend_topk(user_rep, K=10, mask_ids=anime_ids_for_user)
    titles = [id2title.get(aid, f"Unknown ID {aid}") for aid in top10]
    
    print("Top-10 recommendations for synthetic user:", top10)
    print(titles)

Loading model stsb-roberta-large...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 11585.98it/s]
RobertaModel LOAD REPORT from: sentence-transformers/stsb-roberta-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1/1 [00:00<00:00, 69.04it/s]

prompt: A young girl gains magical powers and balances school life with fighting supernatural threats.
prompt_only_user
Top-10 recommendations for synthetic user: [269, 3588, 39417, 14349, 32951, 37984, 36266, 14513, 1535, 38256]
['Bleach', 'Soul Eater', 'Granbelm', 'Little Witch Academia', 'Rokudenashi Majutsu Koushi to Akashic Records', 'Kumo desu ga, Nani ka?', 'Mahou Shoujo Site', 'Magi: The Labyrinth of Magic', 'Death Note', 'Magia Record: Mahou Shoujo Madoka☆Magica Gaiden']
shounen_watcher
Top-10 recommendations for synthetic user: [121, 1535, 269, 29758, 39417, 225, 791, 55453, 1292, 223]
['Fullmetal Alchemist', 'Death Note', 'Bleach', 'Taboo Tattoo', 'Granbelm', 'Dragon Ball GT', 'Arion', 'Naruto (2023)', 'Afro Samurai', 'Dragon Ball']
slice_of_life_fan
Top-10 recommendations for synthetic user: [14967, 3604, 7791, 34148, 39026, 17082, 4983, 2993, 4550, 15911]
['Boku wa Tomodachi ga Sukunai Next', 'Hidamari Sketch x 365', 'K-On!!', 'Nyanko Days', 'Dumbbell Nan Kilo Moteru?', 'A

In [28]:
from sentence_transformers import SentenceTransformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "stsb-roberta-large" # We need to use the same model as the one used to generate the description embeddings, otherwise the embeddings won't be compatible with the model's attention layers.


# Load model
print(f"Loading model {model_name}...")
embedding_model = SentenceTransformer(model_name)
embedding_model = embedding_model.to(device)

# Because we used SentenceTransformer embeddings to train,
# we can in theory use natural language prompts to generate user embeddings without needing to specify anime IDs. 
# This is a powerful feature of using text-based embeddings and attention mechanisms, as it allows us to capture user preferences in a more flexible way.

# Prepare prompt
prompt = "A psychological story where characters are pushed to their limits, dealing with trauma, manipulation, and internal conflict. The narrative focuses on mind games, moral ambiguity, and the consequences of human decisions."  # Example prompt describing a user archetype. In practice, you could have more complex prompts or even multiple prompts per user.
prompt_embedding = embedding_model.encode(prompt, normalize_embeddings=True, show_progress_bar=True, convert_to_tensor=True)  # SentenceTransformer


print(f"prompt: {prompt}")
for u, anime_ids_for_user in archetype_users.items():
    print(u)

    # Create augmented user embedding
    user_rep = create_augmented_user_embedding(anime_ids_for_user, prompt_embedding=prompt_embedding, prompt_weight=2.0)

    # Get top-K recommendations
    top10 = recommend_topk(user_rep, K=10, mask_ids=anime_ids_for_user)
    titles = [id2title.get(aid, f"Unknown ID {aid}") for aid in top10]
    
    print("Top-10 recommendations for synthetic user:", top10)
    print(titles)

Loading model stsb-roberta-large...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 10056.06it/s]
RobertaModel LOAD REPORT from: sentence-transformers/stsb-roberta-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1/1 [00:00<00:00, 74.24it/s]

prompt: A psychological story where characters are pushed to their limits, dealing with trauma, manipulation, and internal conflict. The narrative focuses on mind games, moral ambiguity, and the consequences of human decisions.
prompt_only_user
Top-10 recommendations for synthetic user: [30948, 6547, 16498, 33519, 52842, 44961, 26, 5681, 226, 1508]
['Kowabon', 'Angel Beats!', 'Shingeki no Kyojin', 'Koutetsujou no Kabaneri Movie 1: Tsudou Hikari', 'Guanzhao', 'Platinum End', 'Texhnolyze', 'Summer Wars', 'Elfen Lied', 'Sci-fi Harry']
shounen_watcher
Top-10 recommendations for synthetic user: [1535, 121, 269, 46569, 223, 29758, 791, 32379, 225, 384]
['Death Note', 'Fullmetal Alchemist', 'Bleach', 'Jigokuraku', 'Dragon Ball', 'Taboo Tattoo', 'Arion', 'Berserk', 'Dragon Ball GT', 'Gantz']
slice_of_life_fan
Top-10 recommendations for synthetic user: [34148, 7791, 14967, 4550, 3604, 11843, 36945, 17082, 15911, 35478]
['Nyanko Days', 'K-On!!', 'Boku wa Tomodachi ga Sukunai Next', 'Hyakko', 'Hi